# Domain 1 - Additional-antigen coverage forecasting
**NPHCDA Zero-Dose Modelling Platform - update reflecting the Executive Director's request.**

Extends coverage forecasting beyond the four tracers (BCG, Penta1, Penta3, Measles1) to the
additional antigens for which routine DHIS2 data (2021-2025) exist. Two groups, treated differently:

- **Established antigens** (stable multi-year history): OPV3, IPV1, PCV3, Yellow Fever, Men A -
  screened for an **at-risk-of-decline flag** (a projected fall below 80% of the antigen's **2024**
  level; 2024 is the last complete calendar year, so it is a fixed, fully-reported benchmark).
- **Recently introduced antigens** (still scaling up): second IPV dose (IPV2) and Rotavirus
  (Rota1-3). A "% of 2024" decline flag is invalid while a vaccine ramps, so they are monitored for
  **uptake**, not decline.

Method: Prophet time-series per antigen with 80/95% prediction intervals. All figures are model
estimates. (House style: hyphens only; "modelling"/"programme"; acronyms spelled out.)

In [ ]:
!pip -q install prophet pandas matplotlib

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
import logging
for _l in ("prophet", "cmdstanpy"): logging.getLogger(_l).setLevel(logging.ERROR)

## 1. Load and consolidate the DHIS2 data (main antigens + additional antigens)

In [ ]:
# Point these at the dataset folder (Colab: upload the two CSVs first).
MAIN = "dhis2_data_all_states.csv"                 # BCG/Penta/Measles/OPV/PCV
ADDL = "dhis2_data_additional_antigens.csv"        # IPV1/2, Rota1-3, Yellow Fever, Men A
old = pd.read_csv(MAIN); new = pd.read_csv(ADDL)
keys = ["zone", "state", "lga", "period"]
df = old.merge(new.drop(columns=[c for c in ["row_id"] if c in new.columns]), on=keys, how="left")
df["ds"] = pd.to_datetime(df["period"].astype(str).str.strip(), format="%b-%y", errors="coerce")
print("consolidated:", df.shape)

## 2. Antigen groups (Nigeria routine immunization schedule)

In [ ]:
ESTABLISHED = {"OPV3":"opv_3_count","IPV1":"ipv_1_count","PCV3":"pcv_3_count",
               "Yellow Fever":"yellow_fever_count","Men A":"men_a_count"}
RECENTLY_INTRODUCED = {"IPV2":"ipv_2_count","Rota1":"rota_1_count","Rota2":"rota_2_count","Rota3":"rota_3_count"}
# schedule age (for reference): OPV3/IPV1/PCV3/Rota3 - 14 wks; IPV2/Yellow Fever/Men A - 9 months.

## 3. Forecast each antigen (Prophet) and apply the at-risk-of-decline flag

In [ ]:
def forecast_one(col):
    v = pd.to_numeric(df[col], errors="coerce")
    s = df.assign(y=v).dropna(subset=["ds"]).groupby("ds")["y"].sum().reset_index()
    s = s[s["y"] > 0].sort_values("ds")
    if len(s) < 12: return None, None, None
    m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False,
                interval_width=0.95).fit(s)
    fc = m.predict(m.make_future_dataframe(periods=30, freq="MS"))
    base = s[s["ds"].dt.year == 2024]["y"].mean()
    fcf = fc[fc["ds"] > s["ds"].max()]
    return s, fcf, base

rows = []
for label, col in {**ESTABLISHED, **RECENTLY_INTRODUCED}.items():
    s, fcf, base = forecast_one(col)
    if s is None: continue
    kind = "established" if label in ESTABLISHED else "recently introduced"
    minpct = fcf["yhat"].min() / base * 100 if base else np.nan
    flag = ("n/a - scaling up" if kind == "recently introduced"
            else ("AT-RISK-OF-DECLINE" if minpct < 80 else "on track"))
    rows.append({"Antigen": label, "Group": kind, "2024 base doses/mo": round(base),
                 "Min forecast % of 2024": round(minpct, 1), "Early-warning": flag})
summary = pd.DataFrame(rows); summary

## 4. Forecast panel (established get the 80% early-warning line; recently introduced show uptake)

In [ ]:
PANEL = list(ESTABLISHED.items()) + list(RECENTLY_INTRODUCED.items())
fig, axes = plt.subplots(3, 3, figsize=(17, 12))
for ax,(label,col) in zip(axes.ravel(), PANEL):
    s, fcf, base = forecast_one(col)
    if s is None: ax.axis("off"); continue
    est = label in ESTABLISHED
    ax.plot(s["ds"], s["y"]/1000, color="#1F3B57", lw=2, label="observed")
    ax.plot(fcf["ds"], fcf["yhat"]/1000, color="#C0392B", lw=2, ls="--", label="forecast")
    ax.fill_between(fcf["ds"], fcf["yhat_lower"]/1000, fcf["yhat_upper"]/1000, color="#C0392B", alpha=.15)
    if est and base: ax.axhline(base*.8/1000, color="#7B2FBF", ls=":", lw=1.6, label="80% of 2024 (early-warning)")
    ax.set_title(f"{label} [{'established' if est else 'recently introduced - scale-up'}]",
                 fontsize=11, fontweight="bold", color="#1F3B57", loc="left")
    ax.set_ylabel("doses/mo (000s)", fontsize=8); ax.legend(fontsize=7)
    ax.spines[["top","right"]].set_visible(False)
fig.suptitle("Additional antigen forecasts (DHIS2 2021-2025 -> 2027)", fontsize=13, fontweight="bold", color="#1F3B57")
fig.tight_layout(rect=[0,0,1,0.96]); plt.savefig("additional_antigen_forecasts.png", dpi=150, bbox_inches="tight"); plt.show()

## 5. Interpretation
- **Established antigens** are compared to the **2024** level (the last complete year); a projection
  below 80% raises the **at-risk-of-decline early-warning**. This is an early-warning of *decline*,
  **not** coverage of the eligible child population (that requires a population denominator).
- **Recently introduced antigens** are still scaling up, so they are read as **uptake**, not decline.
- Export `summary` for the report and the app's downloads.

In [ ]:
summary.to_csv("D1_additional_antigen_forecast_summary.csv", index=False); print("saved summary + figure")